# ACE example


In [1]:
pwd

'C:\\Users\\nokni\\work\\MHDTurbPy\\Notebooks_Examples'

In [10]:

%load_ext autoreload
%autoreload 2

from pathlib import Path
from joblib import Parallel, delayed
import importlib.util


def _find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "functions").is_dir() and (candidate / "pyspedas").is_dir():
            return candidate
    raise RuntimeError("Could not locate MHDTurbPy root (missing functions/ and pyspedas/).")


root_dir = _find_repo_root(Path.cwd())
path_setup_file = root_dir / "functions" / "path_setup.py"
spec = importlib.util.spec_from_file_location("mhdturbpy_path_setup", path_setup_file)
if spec is None or spec.loader is None:
    raise RuntimeError(f"Could not load path setup from {path_setup_file}")
path_setup = importlib.util.module_from_spec(spec)
spec.loader.exec_module(path_setup)

root_dir = path_setup.ensure_project_paths(
    start=Path.cwd(),
    include_downloading_helpers=True,
    include_anisotropy_toolbox=True,
    include_sc_pos=True,
)


from functions import download_data as download

from functions import  calc_diagnostics as calc
from functions import  TurbPy as turb
from functions import general_functions as func
from functions import  Figures as figs
from functions import  interactive_figs

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Download ACE data


In [11]:


# ============================================================
# 0) Environment / paths
# ============================================================

# Point this to your local CDF library (needed for some CDAS / CDF readers)
cdf_lib_path = "/Applications/cdf/cdf/lib"

# Credentials are optional (public products will still work when available).
# For private FIELDS/SWEAP products, fill these in.
credentials             =      { 'psp':{
                                           'fields': {'username': 'mvelli', 'password': 'flds@psp'},
                                           'sweap' : {'username': 'mvelli', 'password': '2019swe@pd@ta'}}}

# If you truly want no credentials:
# credentials = None


# How many cores to use
n_jobs     = 1


# ============================================================
# 1) Settings (tightly organized by topic)
# ============================================================

# ---- Paths / IO
PATHS = {
        "Data_path"         : Path(root_dir).joinpath('data'),
        "save_destination"  :  Path(root_dir).joinpath('examples').joinpath('downloaded_intervals'),
}

IO = {
                "overwrite_files"               : 1,        # 1 -> overwrite folders if final.pkl exists
                "save_all"                      : True,     # True -> save final.pkl + general.pkl (+ extras)
                "addit_time_around"             : 1,        # hours padded around each interval for loading
                "gap_time_threshold"            : 5,        # legacy (used downstream)
}

# ---- Mission / geometry
MISSION = {
            "sc"                                        : "ACE",
            "in_rtn"                                    : True,                # True -> RTN output for MAG/VEL when available
}

# ---- Interval generation (LEGACY behavior kept)
# NOTE: In generate_intervals(...), "multiple_intervals" behaves as follows:
#   - multiple_intervals == False  -> ONE interval [start_date, end_date]
#   - multiple_intervals == True   -> MANY intervals using Step and duration
INTERVALS = {
    
    "start_date"              : "2025-10-01 00:00",
    "end_date"                : "2025-10-10 00:00",
    
    # if multiple intervals =True, then define duration and step
    "multiple_intervals"      : False,
    "duration"                : "4H",
    "Step"                    : "210min",
}

# ---- Sampling / resampling
SAMPLING = {
    "part_resol"          : 1,             # miliseconds
    "MAG_resol"           : 1,                # miliseconds
    "upsample_low_freq_ts": False, # keep legacy sync behavior
}

# ---- Data acceptance
QC = {
    "must_have_qtn"   : False,
    "Max_par_missing" : 30,         # % threshold used by main_function()
}

# ---- Instrument selection (PSP-specific)
PSP_INSTRUMENTS = {
    # Particle selection logic (must remain identical)
    "particle_mode"               : "spc",      # {"9th_perih_cut","spc","span"}
    "allow_max_SWEAP_distance"    : False,
    "max_SWEAP_distance"          : 0.25,            # au

    # Interval rejection by distance (None -> no distance rejection)
    "max_PSP_dist": None,

    # SPAN product selection (legacy key)
    "span_key": "spi_sf00",

    # Optional Hampel filtering (kept)
    "apply_hampel": False,
    "hampel_params": {"w": 100, "std": 3},

    # Optional Orlando QTN pickle (None -> disabled)
    "orlandos_QTN": None,

}

# ---- Unified MAG noise removal (works for SOLO merged MAG and PSP SCAM)
# This is the preferred key; legacy Mag_SCAM_PSP["noise_flag"] still works.
MAG_NOISE = {
    "Mag_SCM": {
        "use_SCM"   : True,
        "noise_flag": False,
        "noise_removal": {
            'window_size'         : 2**15,
            'avg_length'          : 1,
            'power_threshold'     : 3.0,
            'freq_min'            : 10.0 # in Hz
            # "hampel_wind": 51,
            # "hampel_thresh": 3.5,
        },
    }
}

# ---- Big gap reporting thresholds (preserved)
GAPS = {
    "Big_Gaps": {
        "E_big_gaps": 10,
        "SC_pot_big_gaps": 10,
        "Mag_big_gaps": 10,
        "Par_big_gaps": 10,
        "QTN_big_gaps": 10,
    }
}

# ---- Optional E-field calibration block (download_data.py uses this)
E_FIELD = {
    "E_field": {
        "flag": False,
        "cadence_seconds": 6,
        "fit_interval_minutes": 4,
        "stride_minutes": 0.1,
        "min_correlation": 0.8,
        "keep_sc_pot": False,
    }
}

# ---- Optional SC potential calibration block (download_data.py uses this)
SC_POT = {
    "sc_pot": {
        "flag": False,
        "fit_window": "6000s",
        "mode": "local",        # {"local","global"} (legacy behavior)
        "overlap_ratio": 0.5,
    }
}

# ---- Downstream diagnostics toggles (kept)
DERIVED = {
    "estimate_derived_param": True,
    "rol_window": "20min",

    # optional analysis blocks used in small windows pipeline
    "PSDs"                   : {"flag":  True},
    'estimate_psd_b'         : 1,                             # Estimate magentic field powes spectral density (keep false)
    'estimate_psd_v'         : 1,                             # Estimate velocity field powes spectral density (keep false)
    'est_PSD_components'     : 1,
    'smooth_psd'             : False,
    "struc_funcs"    : {"flag": False},
                        "npt_struc_funcs": {
                            "flag"                  : False,
                            "five_points_sfunc"     : True,
                            "return_Bmod"           : True,
                            "dt_step"               : 0.25,
                            "est_sfuncs"            : False,
                            "max_qorder"            : 8,
                        },
    'coherence_analysis' : {'flag'                  : False,
                            'method'                : {'min_var': ['B'], 
                                                       'TN_only': ['B', 'E']},
                            'nv'                    : 8,
                            'alpha'                 : 3,
                            'per_thresh'            : 80,
                            'par_thresh'            : 10,
                            'coh_th'                : 0.7,
                            'num_efoldings'         : 3,
                            'njobs'                 : 10,
                            'est_mod'               : True,
                            'estimate_local_V'      : False,
                            
                            'do_coherence_analysis' : True,
                            'estimate_PSDs'         : True,
                            'estimate_coh_coeffs'   : True,
                            'estimate_comp'         : True,
                            'return_coeffs'         : False,
                            'est_sfuncs'            : False,
                            'max_qorder'            : 8
    },

    "use_hampel"     : False,
    "hampel_params"  : {"w": 200, "std": 3},

    "cut_in_small_windows": {
        "flag": False,
        "njobs": 1,
        "Step": "5s",
        "duration": "30s",
    },
}

# ---- Assemble flat settings dict (what the pipeline expects)
settings = {
    **PATHS,
    **IO,
    **MISSION,
    **INTERVALS,
    **SAMPLING,
    **QC,
    **PSP_INSTRUMENTS,
    **MAG_NOISE,
    **GAPS,
    **E_FIELD,
    **SC_POT,
    **DERIVED,
}


# ============================================================
# 2) Variables to download (None means "use defaults" inside PSP.py)
# ============================================================
if settings['sc'] == "PSP":
    
    if settings['in_rtn']:
        vars_2_downnload = {
                            'mag'    : None, 
                            'span'   : None,
                            'span-a' : None,
                            'spc'    : None, 
                            'qtn'    : None,
                            'E_field': settings['E_field']['flag'],
                            'sc_pot' : settings['sc_pot']['flag'],
                            'ephem'  : None}
    else:
        vars_2_downnload        = {  
                                    'mag'     : ['B_SC'],
                                    'span'    : ['DENS',  'VEL_SC', 'TEMP' , 'SUN_DIST'],
                                    'span-a'  : None,
                                    'spc'     : ['np_moment', 'wp_moment', 'vp_moment_SC','sc_pos_HCI'],
                                    'qtn'     : ['electron_density'],
            
                                    'E_field' : settings['E_field']['flag'],
                                    'sc_pot'  : settings['sc_pot']['flag'],
            
            
                                    'ephem'   : None}      

elif settings['sc'] == "SOLO":
    vars_2_downnload = {
                        'mag'    : None,
                        'swa'    : None, 
                        'rpw'    : None, # Default is 'bia-density-10-seconds', but  'bia-density' is also available and probaly interesting
                        'ephem'  : None} 
elif settings['sc'] == "ACE":
    vars_2_downnload = {
                        "mag": {"datatype": "h0"},
                        "par": {"datatype": "h0"},   # or "swe": None
    }
else:
    
    print('Not ready yet!')
    

# ============================================================
# 3) Generate intervals + run
# ============================================================
generated_interval_list = download.generate_intervals(
    settings["start_date"],
    settings["end_date"],
    settings["multiple_intervals"],
    data_path=settings["Data_path"],
    settings=settings,
)

save_path = Path(settings["save_destination"]).joinpath(settings["sc"])
save_path.mkdir(parents=True, exist_ok=True)

Parallel(n_jobs=n_jobs)(
    delayed(download.download_files)(
        jj,
        generated_interval_list,
        settings,
        vars_2_downnload,
        cdf_lib_path,
        credentials,
        save_path,
    )
    for jj in range(len(generated_interval_list))
)


02-Mar-26 06:42:32: Generating only one interval based on the provided start and end times.
02-Mar-26 06:42:32: Start Time: 2025-10-01 00:00:00
End Time: 2025-10-10 00:00:0000

02-Mar-26 06:42:32: Considering a single interval spanning: 2025-10-01 00:00:00 to 2025-10-10 00:00:00
Overwriting folder C:\Users\nokni\work\MHDTurbPy\examples\downloaded_intervals\ACE\2025-10-01_00-00-00_2025-10-10_00-00-00_sc_0
02-Mar-26 06:42:32: Downloading remote index: https://spdf.gsfc.nasa.gov/pub/data/ace/mag/level_2_cdaweb/mfi_h0/2025/0_00-00-00_sc_0



=== ACE loader summary ===
Requested interval : 2025-10-01 00:00:00 -> 2025-10-10 00:00:00
Expanded interval  : 2025-10-01 00:00:00 -> 2025-10-10 00:00:00
Target cadences    : MAG=0.001s, PAR=0.001s
Velocity frame key : ace_frame=GSE (only used if we must manufacture V-components)


02-Mar-26 06:42:33: File is current: ace_data/mag/level_2_cdaweb/mfi_h0/2025/ac_h0_mfi_20251001_v07.cdf
02-Mar-26 06:42:33: File is current: ace_data/mag/level_2_cdaweb/mfi_h0/2025/ac_h0_mfi_20251002_v07.cdf
4: File is current: ace_data/mag/level_2_cdaweb/mfi_h0/2025/ac_h0_mfi_20251003_v07.cdf

02-Mar-26 06:42:34: File is current: ace_data/mag/level_2_cdaweb/mfi_h0/2025/ac_h0_mfi_20251004_v07.cdf
5: File is current: ace_data/mag/level_2_cdaweb/mfi_h0/2025/ac_h0_mfi_20251005_v07.cdf

02-Mar-26 06:42:35: File is current: ace_data/mag/level_2_cdaweb/mfi_h0/2025/ac_h0_mfi_20251006_v07.cdf
6: File is current: ace_data/mag/level_2_cdaweb/mfi_h0/2025/ac_h0_mfi_20251007_v07.cdf

02-Mar-26 06:42:36: File is current: ace_data/mag/level_2_cdaweb/mfi_h0/2025/ac_h0_mfi_20251008_v07.cdf
7: File is current: ace_data/mag/level_2_cdaweb/mfi_h0/2025/ac_h0_mfi_20251009_v07.cdf

02-Mar-26 06:42:41: Downloading remote index: https://spdf.gsfc.nasa.gov/pub/data/ace/swepam/level_2_cdaweb/swe_h0/2025/


0 : Magnitude
1 : BGSEc
2 : BGSM
3 : dBrms
4 : SC_pos_GSE
5 : SC_pos_GSM
--- MAG ---
Source            : pyspedas:ace.mfi:h0
Returned coverage : 2025-10-01 00:00:07 -> 2025-10-09 23:59:49
Columns           : ['Bx', 'By', 'Bz']
--- PAR ---
CDAWeb skipped     : requested start 2025-10-01 00:00:00 is after 2024-07-09 23:59:59


02-Mar-26 06:42:41: Remote index not found: https://spdf.gsfc.nasa.gov/pub/data/ace/swepam/level_2_cdaweb/swe_h0/2025/
02-Mar-26 06:42:43: Skipping remote index: https://spdf.gsfc.nasa.gov/pub/data/ace/swepam/level_2_cdaweb/swe_h0/2025/ (previous attempt failed)

02-Mar-26 06:42:43: Skipping remote index: https://spdf.gsfc.nasa.gov/pub/data/ace/swepam/level_2_cdaweb/swe_h0/2025/ (previous attempt failed)

02-Mar-26 06:42:43: Skipping remote index: https://spdf.gsfc.nasa.gov/pub/data/ace/swepam/level_2_cdaweb/swe_h0/2025/ (previous attempt failed)

02-Mar-26 06:42:43: Skipping remote index: https://spdf.gsfc.nasa.gov/pub/data/ace/swepam/level_2_cdaweb/swe_h0/2025/ (previous attempt failed)

02-Mar-26 06:42:43: Downloading remote index: https://spdf.gsfc.nasa.gov/pub/data/ace/swepam/level_2_cdaweb/swe_k0/2025/
4: File is current: ace_data/swepam/level_2_cdaweb/swe_k0/2025/ac_k0_swe_20251001_v01.cdfb/swe_k0/2025/

02-Mar-26 06:42:44: File is current: ace_data/swepam/level_2_cdaweb/swe_k0/

0 : Np
1 : Vp
2 : He_ratio
3 : Tpr
--- PAR ---
Source            : pyspedas:ace.swe:k0
Returned coverage : 2025-10-01 00:00:00 -> 2025-10-09 23:55:00
Vector columns    : ['Vx', 'Vy', 'Vz']
Vector note       : manufactured only because fallback had only Vp; used V*=Vp/sqrt(3)
Thermals          : Tp[eV], Tp_K[K], Vth[km/s] computed (WIND-consistent)
Key columns       : ['Tp', 'Tp_K', 'Vp', 'Vth', 'Vx', 'Vy', 'Vz', 'np']
--- Cadence check (selected interval) ---
MAG native cadence : ~16.000s (median Δt over selected interval)
MAG note           : requested cadence 0.001s is HIGHER than native
PAR native cadence : ~300.000s (median Δt over selected interval)
PAR note           : requested cadence 0.001s is HIGHER than native
--- Diagnostics ---
Mag fraction missing: 0.0
Par fraction missing: 3.5108024691358026
=== ACE loader done ===


02-Mar-26 06:42:47: Obtained JPL HORIZONS location for ACE (spacecraft) (-92)


INFO: Obtained JPL HORIZONS location for ACE (spacecraft) (-92) [sunpy.coordinates.ephemeris]
Index(['Bx', 'By', 'Bz', 'Vp', 'np', 'Tp', 'Vx', 'Vy', 'Vz', 'Tp_K', 'Vth',
       'TEMP', 'Dist_km', 'x_km', 'y_km', 'z_km', 'Bx_mean', 'By_mean',
       'Bz_mean', 'Vx_mean', 'Vy_mean', 'Vz_mean', 'np_mean'],
      dtype='object')
0 out of 1 finished'Bz', 'Vp', 'np', 'Tp', 'Vx', 'Vy', 'Vz', 'Tp_K', 'Vth',
       'TEMP', 'Dist_km', 'x_km', 'y_km', 'z_km', 'Bx_mean', 'By_mean',
       'Bz_mean', 'Vx_mean', 'Vy_mean', 'Vz_mean', 'np_mean'],
      dtype='object')



[None]

## Visualize downloaded interval


In [12]:
# run_interactive_interval.py
#
# Minimal driver for MHDTurbPy interval visualization.

from importlib import reload
from pathlib import Path

import matplotlib
matplotlib.use("TkAgg")  # notebook: use %matplotlib tk

import matplotlib.pyplot as plt

import interactive_figs as ifigs
import general_functions as func

reload(ifigs)
plt.close("all")

ROOT = Path(r"C:\Users\nokni\work\MHDTurbPy")  # <-- edit this once

SC = ["WIND", "ACE"]  # sc[0] used by flow_mode="sc1_v", sc[1] by "sc2_v"

cfg = dict(
    sc=list(SC),
    which_int=0,
    load_path={s: ROOT / "examples" / "downloaded_intervals" / s for s in SC},
    my_dir=ROOT / "examples",
    save_path=(ROOT / "examples" / "selected_intervals"),
    load_files_func=func.load_files,

    rolling="10s",
    align_intervals_to_first_sc=True,

    enable_flow_separation=True,
    flow_mode="sc1_v",
    flow_v_smooth_window="2min",
    flow_dir_gse=(-1.0, 0.0, 0.0),  # fallback only (used if V is missing/invalid)
    vsw_fallback=400.0,

    alternate_legend_sides=True,
)

cfg["save_path"].mkdir(parents=True, exist_ok=True)

fig, events = ifigs.interactive_mhdturbpy_interval(**cfg)
plt.show()


ERROR: See the raw output from the JPL HORIZONS query at https://ssd.jpl.nasa.gov/api/horizons.api?format=text&EPHEM_TYPE=VECTORS&OUT_UNITS=AU-D&COMMAND=%22-8%22&CSV_FORMAT=%22YES%22&REF_PLANE=ECLIPTIC&REF_SYSTEM=ICRF&TP_TYPE=ABSOLUTE&VEC_LABELS=YES&VEC_CORR=%22NONE%22&VEC_DELTA_T=NO&OBJ_DATA=YES&CENTER=%27500%4010%27&START_TIME=%222025-10-01+00%3A01%3A14.182%22&STOP_TIME=%222025-10-10+00%3A01%3A05.182%22&STEP_SIZE=%2265s%22 [sunpy.coordinates.ephemeris]
02-Mar-26 08:20:09: See the raw output from the JPL HORIZONS query at https://ssd.jpl.nasa.gov/api/horizons.api?format=text&EPHEM_TYPE=VECTORS&OUT_UNITS=AU-D&COMMAND=%22-8%22&CSV_FORMAT=%22YES%22&REF_PLANE=ECLIPTIC&REF_SYSTEM=ICRF&TP_TYPE=ABSOLUTE&VEC_LABELS=YES&VEC_CORR=%22NONE%22&VEC_DELTA_T=NO&OBJ_DATA=YES&CENTER=%27500%4010%27&START_TIME=%222025-10-01+00%3A01%3A14.182%22&STOP_TIME=%222025-10-10+00%3A01%3A05.182%22&STEP_SIZE=%2265s%22
02-Mar-26 08:20:09: C:\Users\nokni\work\MHDTurbPy\functions\interactive_figs.py:3: DeprecationWarni

ValueError: Query failed without known error message; received the following response:
API VERSION: 1.2
API SOURCE: NASA/JPL Horizons API

*******************************************************************************
 Revised: Feb 13, 2026          Wind Spacecraft  / (Earth)                   -8
                                 https://wind.nasa.gov/
                          https://hpde.io/SMWG/Observatory/Wind
https://www.nasa.gov/feature/goddard/2019/25-years-of-science-in-the-solar-wind

 BACKGROUND:

  The primary science objectives of the Wind mission are:
   - Provide plasma, energetic particle, and magnetic field data for 
      magnetospheric and ionospheric studies
   - Investigate plasma processes occurring in the near-Earth solar wind
   - Provide baseline, 1 AU, ecliptic plane observations for inner and outer 
      heliospheric missions

  Wind is a spin-stabilized spacecraft launched on a Delta II 7925-10 rocket
  1994-Nov-1 @ 09:31 UTC from pad 17B at Cape Canaveral,